# DocSight RAG — Evaluation Notebook
Runs RAGAS evaluation and ablation study, then visualizes results.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import config

## Step 1 — Generate Evaluation Dataset

In [ ]:
# Run only if eval files don't exist yet
import os
if not config.EVAL_FINANCE_PATH.exists() or not config.EVAL_MEDICAL_PATH.exists():
    from src.evaluation.generate_qa_pairs import generate_eval_set
    import json
    config.EVAL_DIR.mkdir(parents=True, exist_ok=True)

    for domain, path in [('finance', config.EVAL_FINANCE_PATH), ('medical', config.EVAL_MEDICAL_PATH)]:
        data = generate_eval_set(domain, n=100)
        with open(path, 'w') as f:
            json.dump(data, f, indent=2)
        print(f'{domain}: {len(data)} eval pairs generated')
else:
    print('Eval files already exist.')

## Step 2 — Run RAGAS Evaluation

In [ ]:
from src.evaluation.run_ragas import evaluate_domain

df_fin = evaluate_domain('finance', sample=50)  # sample=50 is faster for demo
df_med = evaluate_domain('medical', sample=50)

In [ ]:
# Visualize RAGAS scores
metrics = ['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']
targets_min  = [0.80, 0.75, 0.70, 0.65]
targets_good = [0.90, 0.85, 0.80, 0.75]

fin_scores = [df_fin[m].mean() for m in metrics]
med_scores = [df_med[m].mean() for m in metrics]

x = np.arange(len(metrics))
w = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(x - w, fin_scores, w, label='Finance', color='#2196F3')
ax.bar(x,      med_scores, w, label='Medical', color='#4CAF50')
ax.plot(x, targets_min,  '--', color='orange', label='Min target')
ax.plot(x, targets_good, '--', color='green',  label='Good target')

ax.set_xticks(x)
ax.set_xticklabels([m.replace('_', ' ').title() for m in metrics])
ax.set_ylim(0, 1.0)
ax.set_ylabel('Score')
ax.set_title('RAGAS Evaluation — DocSight RAG')
ax.legend()
plt.tight_layout()
plt.savefig('../data/eval/ragas_scores.png', dpi=150)
plt.show()

## Step 3 — Ablation Study

In [ ]:
from src.evaluation.ablation import run_ablation

abl_fin = run_ablation('finance', n=30)
abl_med = run_ablation('medical', n=30)

print('\nFinance ablation:')
print(abl_fin)
print('\nMedical ablation:')
print(abl_med)

In [ ]:
# Ablation bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, df, title in [(axes[0], abl_fin, 'Finance'), (axes[1], abl_med, 'Medical')]:
    df_plot = df.set_index('strategy')
    df_plot[['context_precision', 'context_recall']].plot(
        kind='bar', ax=ax, color=['#2196F3', '#4CAF50'], rot=0
    )
    ax.set_title(f'{title} — BM25 vs Dense vs Hybrid')
    ax.set_ylim(0, 1.0)
    ax.set_ylabel('Score')
    ax.legend(['Context Precision', 'Context Recall'])

plt.tight_layout()
plt.savefig('../data/eval/ablation.png', dpi=150)
plt.show()

## Summary Table

In [ ]:
summary = pd.DataFrame({
    'Domain':    ['Finance', 'Medical'],
    'Faithfulness':       [df_fin['faithfulness'].mean(), df_med['faithfulness'].mean()],
    'Answer Relevancy':   [df_fin['answer_relevancy'].mean(), df_med['answer_relevancy'].mean()],
    'Context Precision':  [df_fin['context_precision'].mean(), df_med['context_precision'].mean()],
    'Context Recall':     [df_fin['context_recall'].mean(), df_med['context_recall'].mean()],
})
summary.set_index('Domain').round(3).style.background_gradient(cmap='RdYlGn', vmin=0.5, vmax=1.0)